# 📍 Notebook 04: Embeddings & Positional Encoding

## Learning Objectives

By the end of this notebook, you will:
1. Understand why transformers need explicit position information (permutation invariance)
2. Implement token embeddings with `nn.Embedding` and visualize similarity structure
3. Build sinusoidal positional encodings from the original "Attention Is All You Need" paper
4. Implement Rotary Position Embeddings (RoPE) used in LLaMA, Gemma, and modern LLMs
5. Compare sinusoidal vs RoPE vs learned embeddings and understand the tradeoffs

## Prerequisites
- Notebook 01: MLX Fundamentals (arrays, `nn.Module`)
- Notebook 02: Math Foundations (dot products, cosine similarity)
- Notebook 03: Tokenization (token IDs as integers)

## ✅ Environment Validation

In [1]:
from utils.checks import validate_environment, print_environment_report
validate_environment()
print_environment_report()

  Environment Validation Report
  Platform : macOS-26.4.1-arm64-arm-64bit-Mach-O
  Chip     : Apple M4 Pro
  Python   : 3.13.13  ✅
  MLX      : 0.31.1  ✅
  Metal GPU: available  ✅
  Memory   : 48.0 GB  ✅


{'python_version': '3.13.13',
 'python_ok': True,
 'mlx_available': True,
 'mlx_version': '0.31.1',
 'metal_available': True,
 'memory_gb': 48.0,
 'memory_ok': True,
 'chip': 'Apple M4 Pro',
 'platform': 'macOS-26.4.1-arm64-arm-64bit-Mach-O'}

## 🤔 Why Do Transformers Need Position Information?

Here’s the core problem: **transformers are permutation-invariant**.

An RNN processes tokens left-to-right — position is baked into the sequential computation. Token 5 is processed *after* tokens 1–4, so the hidden state naturally encodes order.

A transformer processes all tokens **in parallel** via self-attention. The attention operation is just weighted sums of value vectors — it treats the input as a **set**, not a sequence. Without position info:

> "The cat sat on the mat" and "mat the on sat cat the" produce **identical** attention outputs.

💡 **Key insight**: We must *inject* position information into the input embeddings so the model can distinguish "cat" at position 1 from "cat" at position 50.

Let’s prove this with code.

### 📦 Library Imports

The next cell loads the libraries we need for this section. Don't worry about memorizing every import — just run the cell and move on. We'll explain each library as we use it.

In [2]:
import mlx.core as mx
import mlx.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"figure.dpi": 120, "font.size": 10})

# Prove: attention is permutation-invariant
# Simple dot-product attention: softmax(Q @ K^T) @ V
def simple_attention(x):
    """Minimal self-attention (no learned weights, just raw dot products)."""
    scores = x @ x.T  # shape: (seq_len, d_model) @ (d_model, seq_len) -> (seq_len, seq_len)
    weights = mx.softmax(scores, axis=-1)  # shape: (seq_len, seq_len)
    output = weights @ x  # shape: (seq_len, seq_len) @ (seq_len, d_model) -> (seq_len, d_model)
    return output

# Create a small "sentence" of 4 token embeddings
mx.random.seed(42)
x = mx.random.normal((4, 8))  # shape: (4, 8) — 4 tokens, 8-dim embeddings
mx.eval(x)

# Original order
out_original = simple_attention(x)  # shape: (4, 8)

# Shuffle the tokens: [0,1,2,3] -> [2,0,3,1]
perm = [2, 0, 3, 1]
x_shuffled = x[perm]  # shape: (4, 8)
out_shuffled = simple_attention(x_shuffled)  # shape: (4, 8)

# Un-shuffle the output to compare
inv_perm = [1, 3, 0, 2]  # inverse of [2,0,3,1]
out_unshuffled = out_shuffled[inv_perm]  # shape: (4, 8)

mx.eval(out_original, out_unshuffled)
diff = mx.abs(out_original - out_unshuffled).max().item()
print(f"Max difference between original and permuted-then-unpermuted: {diff:.2e}")
print(f"Identical (within float tolerance)? {diff < 1e-5}")
print()
print("⚠️ This proves attention doesn't care about token ORDER.")
print("   Without positional encoding, 'cat sat mat' == 'mat cat sat' to the model.")

Max difference between original and permuted-then-unpermuted: 1.19e-07
Identical (within float tolerance)? True

⚠️ This proves attention doesn't care about token ORDER.
   Without positional encoding, 'cat sat mat' == 'mat cat sat' to the model.


---
## Part 1: Token Embeddings

Before we add position info, let’s understand the base layer: **token embeddings**.

An embedding table is just a matrix of shape `(vocab_size, d_model)`. Each row is a learnable vector for one token. "Looking up" a token is just indexing into this matrix — no multiplication needed.

💡 **Key insight**: The embedding table is the model’s entire "vocabulary knowledge." During training, similar tokens ("run", "running", "ran") get pushed toward similar vectors.

In [3]:
# Token embedding lookup with nn.Embedding
vocab_size = 128   # small vocab for demo
d_model = 32       # embedding dimension

# Create embedding table
embed = nn.Embedding(vocab_size, d_model)

# Simulate token IDs (like output from a tokenizer)
token_ids = mx.array([10, 25, 10, 42, 77, 25])  # shape: (6,)
# Note: token 10 appears twice, token 25 appears twice

# Lookup: just indexing into the table
embeddings = embed(token_ids)  # shape: (6, 32)
mx.eval(embeddings)

print(f"Vocab size: {vocab_size}")
print(f"Embedding dim (d_model): {d_model}")
print(f"Embedding table shape: ({vocab_size}, {d_model})")
print(f"Token IDs: {token_ids.tolist()}")
print(f"Output shape: {embeddings.shape}")  # (6, 32)
print(f"\nParameters: {vocab_size * d_model:,} floats = {vocab_size * d_model * 4 / 1024:.1f} KB (float32)")
print(f"\n✅ Same token ID -> same embedding vector:")
print(f"   token_ids[0] == token_ids[2] == 10: vectors equal? {mx.array_equal(embeddings[0], embeddings[2]).item()}")
print(f"   token_ids[1] == token_ids[5] == 25: vectors equal? {mx.array_equal(embeddings[1], embeddings[5]).item()}")

Vocab size: 128
Embedding dim (d_model): 32
Embedding table shape: (128, 32)
Token IDs: [10, 25, 10, 42, 77, 25]
Output shape: (6, 32)

Parameters: 4,096 floats = 16.0 KB (float32)

✅ Same token ID -> same embedding vector:
   token_ids[0] == token_ids[2] == 10: vectors equal? True
   token_ids[1] == token_ids[5] == 25: vectors equal? True


### ⚠️ Handling Common Errors

When working with ML code, errors are normal and expected. Here's a pattern for handling them gracefully — if something goes wrong, you get a helpful message instead of a crash.

In [4]:
# 💡 Error handling pattern — use this when operations might fail
try:
    # This is where your ML code goes
    import mlx.core as mx
    test = mx.array([1.0, 2.0, 3.0])
    result = mx.sum(test)
    mx.eval(result)
    print(f'✅ Success! Result: {result.item()}')
except Exception as e:
    print(f'❌ Error: {e}')
    print('💡 Tip: Check that MLX is installed and your inputs are valid')

✅ Success! Result: 6.0


### Visualizing Embedding Vectors & Cosine Similarity

Let’s look at what the raw embedding vectors look like, then compute a cosine similarity matrix to see which tokens are "close" in embedding space.

🎯 **Interview tip**: Cosine similarity is the standard way to measure embedding closeness. It’s invariant to vector magnitude — only direction matters.

In [5]:
# Pick some token IDs to compare
test_ids = mx.array([10, 11, 12, 50, 51, 52, 100, 101, 102])  # shape: (9,)
test_embeds = embed(test_ids)  # shape: (9, 32)
mx.eval(test_embeds)

# Cosine similarity: dot(a, b) / (||a|| * ||b||)
norms = mx.sqrt(mx.sum(test_embeds * test_embeds, axis=-1, keepdims=True))  # shape: (9, 1)
normalized = test_embeds / norms  # shape: (9, 32)
cosine_sim = normalized @ normalized.T  # shape: (9, 9)
mx.eval(cosine_sim)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: raw embedding vectors as heatmap
im0 = axes[0].imshow(np.array(test_embeds), aspect='auto', cmap='RdBu_r')
axes[0].set_title('Raw Embedding Vectors')
axes[0].set_xlabel('Embedding dimension')
axes[0].set_ylabel('Token ID')
axes[0].set_yticks(range(len(test_ids.tolist())))
axes[0].set_yticklabels([str(t) for t in test_ids.tolist()])
plt.colorbar(im0, ax=axes[0])

# Right: cosine similarity matrix
labels = [str(t) for t in test_ids.tolist()]
im1 = axes[1].imshow(np.array(cosine_sim), cmap='coolwarm', vmin=-1, vmax=1)
axes[1].set_title('Cosine Similarity Matrix')
axes[1].set_xticks(range(len(labels)))
axes[1].set_xticklabels(labels, rotation=45)
axes[1].set_yticks(range(len(labels)))
axes[1].set_yticklabels(labels)
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

print("\n💡 At initialization, embeddings are random — no meaningful clusters yet.")
print("   After training, semantically similar tokens will cluster together.")


💡 At initialization, embeddings are random — no meaningful clusters yet.
   After training, semantically similar tokens will cluster together.


/var/folders/gg/dfm95f2x2llgzc8_3fnnxxzc0000gp/T/ipykernel_46338/1719668089.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Part 2: Sinusoidal Positional Encoding

The original transformer paper ("Attention Is All You Need", 2017) proposed a clever fixed encoding using sine and cosine waves at different frequencies:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

Where `pos` is the position in the sequence and `i` is the dimension index.

💡 **Intuition**: Each dimension oscillates at a different frequency. Low dimensions change fast (high frequency), high dimensions change slow (low frequency). Together, they create a unique "fingerprint" for each position — like binary counting but with smooth waves.

⚡ **Why fixed, not learned?** Fixed sinusoidal encodings can generalize to sequence lengths not seen during training. Learned embeddings can’t.

In [6]:
# Step-by-step sinusoidal positional encoding

def sinusoidal_pe(max_seq_len: int, d_model: int) -> mx.array:
    """Compute sinusoidal positional encoding matrix.
    
    Returns: shape (max_seq_len, d_model)
    """
    # Step 1: Position indices
    pos = mx.arange(max_seq_len).reshape(-1, 1)  # shape: (max_seq_len, 1)
    
    # Step 2: Dimension indices (only even: 0, 2, 4, ...)
    i = mx.arange(0, d_model, 2).reshape(1, -1)  # shape: (1, d_model//2)
    
    # Step 3: Compute the angle: pos / 10000^(2i/d_model)
    # Use log-space for numerical stability: exp(-2i/d * ln(10000))
    angle = pos * mx.exp(-i * (mx.log(mx.array(10000.0)) / d_model))  # shape: (max_seq_len, d_model//2)
    
    # Step 4: Apply sin to even indices, cos to odd indices
    pe_sin = mx.sin(angle)  # shape: (max_seq_len, d_model//2)
    pe_cos = mx.cos(angle)  # shape: (max_seq_len, d_model//2)
    
    # Step 5: Interleave sin and cos
    # Stack along last dim then reshape: [sin0, cos0, sin1, cos1, ...]
    pe = mx.zeros((max_seq_len, d_model))
    pe = pe.at[:, 0::2].add(pe_sin)  # even dims get sin
    pe = pe.at[:, 1::2].add(pe_cos)  # odd dims get cos
    
    mx.eval(pe)
    return pe

# Generate PE for 64 positions, 32 dimensions
max_seq_len, d_model = 64, 32
pe = sinusoidal_pe(max_seq_len, d_model)  # shape: (64, 32)
print(f"PE matrix shape: {pe.shape}")
print(f"\nFirst position (pos=0):")
print(f"  sin(0)=0, cos(0)=1 pattern: {pe[0, :6].tolist()}")
print(f"\nPosition 1 vs Position 2 (should differ):")
print(f"  pos=1: {pe[1, :6].tolist()}")
print(f"  pos=2: {pe[2, :6].tolist()}")

PE matrix shape: (64, 32)

First position (pos=0):
  sin(0)=0, cos(0)=1 pattern: [0.0, 1.0, 0.0, 1.0, 0.0, 1.0]

Position 1 vs Position 2 (should differ):
  pos=1: [0.8414710164070129, 0.5403022766113281, 0.5331684350967407, 0.8460091352462769, 0.3109836280345917, 0.950415313243866]
  pos=2: [0.9092974066734314, -0.416146844625473, 0.9021307229995728, 0.431462824344635, 0.5911272168159485, 0.8065783381462097]


### Visualizing the Sinusoidal PE Matrix

The heatmap below reveals the wave structure. Each column (dimension) oscillates at a different frequency. Each row (position) has a unique pattern.

🎯 **Interview tip**: The sinusoidal PE lets the model learn to attend to relative positions because `PE(pos+k)` can be expressed as a linear function of `PE(pos)` for any fixed offset `k`.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: full PE heatmap
pe_np = np.array(pe)
im0 = axes[0].imshow(pe_np, aspect='auto', cmap='RdBu_r')
axes[0].set_title('Sinusoidal Positional Encoding')
axes[0].set_xlabel('Embedding dimension')
axes[0].set_ylabel('Position')
plt.colorbar(im0, ax=axes[0])

# Right: individual dimension waves
for dim in [0, 1, 4, 5, 16, 17]:
    axes[1].plot(pe_np[:, dim], label=f'dim {dim}', alpha=0.8)
axes[1].set_title('PE Values Across Positions (Selected Dims)')
axes[1].set_xlabel('Position')
axes[1].set_ylabel('PE value')
axes[1].legend(fontsize=7, ncol=2)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Show uniqueness: cosine similarity between position encodings
pe_norms = pe_np / np.linalg.norm(pe_np, axis=-1, keepdims=True)
pos_sim = pe_norms @ pe_norms.T  # shape: (64, 64)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(pos_sim, cmap='viridis')
ax.set_title('Position Similarity (Cosine) — Each Position is Unique')
ax.set_xlabel('Position')
ax.set_ylabel('Position')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print("\n💡 Notice: nearby positions are more similar (bright diagonal band).")
print("   This gives the model a smooth notion of 'closeness' in position.")


💡 Notice: nearby positions are more similar (bright diagonal band).
   This gives the model a smooth notion of 'closeness' in position.


/var/folders/gg/dfm95f2x2llgzc8_3fnnxxzc0000gp/T/ipykernel_46338/675152290.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/gg/dfm95f2x2llgzc8_3fnnxxzc0000gp/T/ipykernel_46338/675152290.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Part 3: Rotary Position Embeddings (RoPE)

RoPE (Su et al., 2021) is the positional encoding used in **LLaMA, Gemma, Mistral, and most modern LLMs**. Instead of *adding* position info to embeddings, RoPE *rotates* them.

### The 2D Rotation Intuition

Think of each pair of embedding dimensions as a 2D point. RoPE rotates that point by an angle proportional to its position:

$$\begin{bmatrix} x'_0 \\ x'_1 \end{bmatrix} = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix} \begin{bmatrix} x_0 \\ x_1 \end{bmatrix}$$

Where $\theta = pos \cdot \omega_i$ and $\omega_i = 1 / 10000^{2i/d}$.

The magic: when you compute `dot(RoPE(q, pos_m), RoPE(k, pos_n))`, the result depends only on the **relative distance** `m - n`, not the absolute positions. This is exactly what we want for attention!

💡 **Key insight**: Rotation preserves vector norms (`||RoPE(x)|| == ||x||`), so it doesn’t distort the embedding magnitudes — it only encodes position through the *angle*.

⚡ **Apple Silicon note**: RoPE is applied to Q and K *after* projection, not to the raw embeddings. This means it’s computed per-head, which parallelizes well on Metal.

### 🕐 The Clock Hand Analogy

Before we dive into code, here's the plain-English version:

**RoPE rotates vectors like a clock hand — the angle tells you the position.**

Imagine each pair of embedding dimensions as a clock face. At position 0, the hand points at 12 o'clock. At position 1, it rotates a little. At position 2, a little more. The key trick: different dimension pairs rotate at different speeds — some are "second hands" (fast), others are "hour hands" (slow). Together, they give every position a unique combination of angles, like a timestamp.

Why does this help attention? When two tokens compute their attention score (a dot product), the rotation angles *subtract*. So the score depends on how far apart the tokens are (relative position), not where they sit in the sequence (absolute position). Two tokens 5 apart always "see" the same angular difference, whether they're at positions 3 & 8 or 100 & 105.

That's the entire idea. Now let's build it step by step.

In [8]:
# Step 1: Precompute rotation frequencies

def precompute_freqs_cis(d_head: int, max_seq_len: int, theta: float = 10000.0) -> tuple:
    """Precompute cos and sin for RoPE.
    
    Args:
        d_head: head dimension (must be even)
        max_seq_len: maximum sequence length
        theta: base frequency (10000 in original paper)
    
    Returns:
        cos_freqs: shape (max_seq_len, d_head//2)
        sin_freqs: shape (max_seq_len, d_head//2)
    """
    # Frequency for each pair of dimensions
    i = mx.arange(0, d_head, 2)  # shape: (d_head//2,)
    freqs = 1.0 / (theta ** (i / d_head))  # shape: (d_head//2,)
    
    # Outer product: position * frequency
    pos = mx.arange(max_seq_len)  # shape: (max_seq_len,)
    angles = pos.reshape(-1, 1) * freqs.reshape(1, -1)  # shape: (max_seq_len, d_head//2)
    
    cos_freqs = mx.cos(angles)  # shape: (max_seq_len, d_head//2)
    sin_freqs = mx.sin(angles)  # shape: (max_seq_len, d_head//2)
    
    mx.eval(cos_freqs, sin_freqs)
    return cos_freqs, sin_freqs

# Demo
d_head = 16
max_seq = 64
cos_f, sin_f = precompute_freqs_cis(d_head, max_seq)
print(f"cos_freqs shape: {cos_f.shape}")  # (64, 8)
print(f"sin_freqs shape: {sin_f.shape}")  # (64, 8)
print(f"\nFrequencies for d_head={d_head}:")
freqs = 1.0 / (10000.0 ** (mx.arange(0, d_head, 2) / d_head))
mx.eval(freqs)
print(f"  {freqs.tolist()}")
print(f"  Fastest (dim 0): completes full rotation every {2 * 3.14159 / freqs[0].item():.1f} positions")
print(f"  Slowest (dim {d_head-2}): completes full rotation every {2 * 3.14159 / freqs[-1].item():.1f} positions")

cos_freqs shape: (64, 8)
sin_freqs shape: (64, 8)

Frequencies for d_head=16:
  [1.0, 0.3162277638912201, 0.09999999403953552, 0.03162277862429619, 0.009999999776482582, 0.0031622773967683315, 0.0010000000474974513, 0.0003162277862429619]
  Fastest (dim 0): completes full rotation every 6.3 positions
  Slowest (dim 14): completes full rotation every 19869.2 positions


### Applying RoPE to Query/Key Vectors

RoPE rotates each consecutive pair of dimensions `(x_0, x_1)`, `(x_2, x_3)`, etc. by the position-dependent angle. This is applied to Q and K vectors *after* the linear projection.

⚠️ **Common pitfall**: RoPE is applied to Q and K only, **not** to V. Values carry content, not position.

In [9]:
# Step 2: Apply RoPE rotation

def apply_rope(x: mx.array, cos_freqs: mx.array, sin_freqs: mx.array) -> mx.array:
    """Apply Rotary Position Embeddings.
    
    Args:
        x: shape (batch, seq_len, n_heads, d_head)
        cos_freqs: shape (seq_len, d_head//2)
        sin_freqs: shape (seq_len, d_head//2)
    
    Returns:
        shape (batch, seq_len, n_heads, d_head) — rotated tensor
    """
    seq_len = x.shape[1]
    d_head = x.shape[-1]
    
    # Reshape x into pairs: (..., d_head//2, 2)
    x_pairs = x.reshape(*x.shape[:-1], d_head // 2, 2)  # shape: (B, T, H, d_head//2, 2)
    x_0 = x_pairs[..., 0]  # shape: (B, T, H, d_head//2)
    x_1 = x_pairs[..., 1]  # shape: (B, T, H, d_head//2)
    
    # Slice frequencies to match sequence length and broadcast
    cos_f = cos_freqs[:seq_len].reshape(1, seq_len, 1, -1)  # shape: (1, T, 1, d_head//2)
    sin_f = sin_freqs[:seq_len].reshape(1, seq_len, 1, -1)  # shape: (1, T, 1, d_head//2)
    
    # Apply 2D rotation to each pair
    rotated_0 = x_0 * cos_f - x_1 * sin_f  # shape: (B, T, H, d_head//2)
    rotated_1 = x_0 * sin_f + x_1 * cos_f  # shape: (B, T, H, d_head//2)
    
    # Reassemble: stack pairs and reshape back
    rotated = mx.stack([rotated_0, rotated_1], axis=-1)  # shape: (B, T, H, d_head//2, 2)
    return rotated.reshape(x.shape)  # shape: (B, T, H, d_head)

# Demo: apply RoPE to a batch of Q vectors
batch, seq_len, n_heads, d_head = 2, 8, 4, 16
mx.random.seed(42)
q = mx.random.normal((batch, seq_len, n_heads, d_head))  # shape: (2, 8, 4, 16)

cos_f, sin_f = precompute_freqs_cis(d_head, seq_len)
q_rotated = apply_rope(q, cos_f, sin_f)  # shape: (2, 8, 4, 16)
mx.eval(q, q_rotated)

print(f"Input Q shape:  {q.shape}")
print(f"Output Q shape: {q_rotated.shape}")
print(f"\n✅ Shapes preserved — RoPE is a drop-in replacement.")

Input Q shape:  (2, 8, 4, 16)
Output Q shape: (2, 8, 4, 16)

✅ Shapes preserved — RoPE is a drop-in replacement.


### Verifying Norm Preservation

A rotation matrix is orthogonal, so it preserves the L2 norm of every vector. Let’s verify: `||RoPE(x, pos)|| == ||x||` for all positions.

🎯 **Interview tip**: Norm preservation is why RoPE is preferred over additive PE in practice — it doesn’t change the scale of activations, which helps training stability.

In [10]:
# Verify norm preservation: ||RoPE(x)|| == ||x||
norm_before = mx.sqrt(mx.sum(q * q, axis=-1))  # shape: (2, 8, 4)
norm_after = mx.sqrt(mx.sum(q_rotated * q_rotated, axis=-1))  # shape: (2, 8, 4)
mx.eval(norm_before, norm_after)

max_norm_diff = mx.abs(norm_before - norm_after).max().item()
mean_norm_diff = mx.mean(mx.abs(norm_before - norm_after)).item()

print("Norm Preservation Check:")
print(f"  Max  |norm_before - norm_after|: {max_norm_diff:.2e}")
print(f"  Mean |norm_before - norm_after|: {mean_norm_diff:.2e}")
print(f"  Preserved (within float tolerance)? {max_norm_diff < 1e-5}")

# Show a few examples
print(f"\nSample norms (batch=0, head=0):")
for pos in range(min(4, seq_len)):
    nb = norm_before[0, pos, 0].item()
    na = norm_after[0, pos, 0].item()
    print(f"  pos={pos}: before={nb:.6f}, after={na:.6f}, diff={abs(nb-na):.2e}")

print(f"\n✅ RoPE is a pure rotation — norms are perfectly preserved.")

Norm Preservation Check:
  Max  |norm_before - norm_after|: 9.54e-07
  Mean |norm_before - norm_after|: 1.79e-07
  Preserved (within float tolerance)? True

Sample norms (batch=0, head=0):
  pos=0: before=3.260346, after=3.260346, diff=0.00e+00
  pos=1: before=3.915988, after=3.915988, diff=0.00e+00
  pos=2: before=5.052162, after=5.052161, diff=9.54e-07
  pos=3: before=3.989208, after=3.989208, diff=2.38e-07

✅ RoPE is a pure rotation — norms are perfectly preserved.


### Visualizing RoPE Rotation

Let’s see how RoPE rotates embedding pairs in 2D space. Each pair of dimensions gets rotated by a position-dependent angle. Low-index pairs rotate fast, high-index pairs rotate slow.

In [11]:
# Visualize RoPE rotation in 2D for a single vector across positions
d_head_viz = 8
max_pos = 32
cos_v, sin_v = precompute_freqs_cis(d_head_viz, max_pos)

# Fixed input vector (same at every position)
x_fixed = mx.ones((1, max_pos, 1, d_head_viz))  # shape: (1, 32, 1, 8)
x_rotated = apply_rope(x_fixed, cos_v, sin_v)  # shape: (1, 32, 1, 8)
mx.eval(x_rotated)

x_rot_np = np.array(x_rotated[0, :, 0, :])  # shape: (32, 8)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
pair_names = ['Pair 0 (dims 0,1)', 'Pair 1 (dims 2,3)', 'Pair 2 (dims 4,5)', 'Pair 3 (dims 6,7)']

for idx in range(4):
    ax = axes[idx]
    d0, d1 = idx * 2, idx * 2 + 1
    xs = x_rot_np[:, d0]
    ys = x_rot_np[:, d1]
    
    # Color by position
    colors = np.arange(max_pos)
    scatter = ax.scatter(xs, ys, c=colors, cmap='viridis', s=20, zorder=2)
    
    # Connect consecutive positions
    ax.plot(xs, ys, 'k-', alpha=0.2, linewidth=0.5)
    
    # Mark start and end
    ax.scatter([xs[0]], [ys[0]], c='red', s=60, marker='o', zorder=3, label='pos=0')
    ax.scatter([xs[-1]], [ys[-1]], c='blue', s=60, marker='s', zorder=3, label=f'pos={max_pos-1}')
    
    ax.set_title(pair_names[idx], fontsize=9)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.legend(fontsize=7)

plt.suptitle('RoPE: 2D Rotation of Dimension Pairs Across Positions', fontsize=11)
plt.tight_layout()
plt.show()

print("\n💡 Pair 0 (low dims) rotates fast — completes many cycles.")
print("   Pair 3 (high dims) rotates slow — barely moves.")
print("   This multi-frequency structure gives each position a unique encoding.")


💡 Pair 0 (low dims) rotates fast — completes many cycles.
   Pair 3 (high dims) rotates slow — barely moves.
   This multi-frequency structure gives each position a unique encoding.


/var/folders/gg/dfm95f2x2llgzc8_3fnnxxzc0000gp/T/ipykernel_46338/284437083.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---

### 🎯 Interview Question nb04-q01  ·  [warmup]  ·  mle

**Q:** Compare absolute sinusoidal positional encoding (the original Transformer) with learned positional embeddings (GPT-2, BERT). When does each win, and what's the failure mode that eventually killed them both in favour of RoPE/ALiBi?

<details>
<summary>Key points in a strong answer</summary>

- Sinusoidal (Vaswani et al. 2017): `PE(pos, 2i) = sin(pos / 10000^(2i/d))`, `PE(pos, 2i+1) = cos(pos / 10000^(2i/d))`. Zero learnable parameters; deterministic; added ONCE to input embeddings before the first attention block.
- Learned PE (GPT-2, BERT): a trainable `(max_seq_len, d_model)` matrix indexed by position. Costs V·D extra params; tends to fit the training distribution slightly better than sinusoidal at matched data.
- Sinusoidal wins on THEORETICAL extrapolation: the wave structure continues past `max_seq_len`. In practice 2017-era benchmarks showed it still degrades sharply because attention never saw high-frequency components for those positions during training.
- Learned wins on FINITE contexts: a 512-token model that will NEVER see longer inputs at inference is marginally better with learned PE — the optimizer finds positional representations that correlate with the data.
- Failure mode — both: neither encodes RELATIVE position. The model has to derive 'distance between token m and token n' from `PE(m) − PE(n)` inside attention, which is a sub-optimal indirect signal.
- Failure mode — learned: hard ceiling at `max_seq_len`. Position 512 for a 512-trained model is literally uninitialized; the model emits garbage.
- This is why RoPE (2021) and ALiBi (2021) — both RELATIVE schemes — took over. Since LLaMA (2023), no frontier LLM ships with absolute PE.
</details>

> ⚠️ **Trap:** Saying 'sinusoidal extrapolates, so early models could run on arbitrary lengths' — the wave structure extrapolates, but the ATTENTION heads learned only on training-length positions and silently break at inference time just as hard as learned PE does.
>
> 📚 **References:** https://arxiv.org/abs/1706.03762, https://arxiv.org/abs/2104.09864

---

### 🎯 Interview Question nb04-q02  ·  [core]  ·  mle, research_engineer

**Q:** Derive RoPE from first principles. What property do we WANT the position-aware inner product <f(q, m), f(k, n)> to satisfy, why does the complex-number formulation make that derivation trivial, and what exactly is the 'inner-product preserves relative distance' property?

<details>
<summary>Key points in a strong answer</summary>

- Desired property: `<f(q, m), f(k, n)> = g(q, k, m − n)` — the inner product between a rotated query at position m and a rotated key at position n must depend ONLY on the RELATIVE offset m − n, never on the absolute positions.
- Ansatz: try multiplicative (not additive) position encoding. Split the d-dim vector into d/2 pairs and treat each pair as a single complex number: `q_j = q_{2j} + i·q_{2j+1}`. Encode position by MULTIPLYING each complex coordinate by `e^(i·m·θ_j)` where `θ_j = 10000^(−2j/d)`.
- In complex form the inner product of the rotated vectors becomes `Re[ Σ_j (q_j · e^(i·m·θ_j)) · conj(k_j · e^(i·n·θ_j)) ] = Re[ Σ_j q_j · conj(k_j) · e^(i·(m−n)·θ_j) ]` — the absolute m and n vanish; only m − n survives. That's the desired property EXACTLY.
- Real-valued implementation: each complex multiply by `e^(i·m·θ)` is a 2×2 rotation `[cos(mθ), −sin(mθ); sin(mθ), cos(mθ)]` applied to the pair (q_{2j}, q_{2j+1}). d/2 independent 2×2 rotations per token per head.
- Inner-product-preserves-relative-distance property: for any shift Δ, `<RoPE(q, m), RoPE(k, n)> = <RoPE(q, m+Δ), RoPE(k, n+Δ)>`. Whiteboard check: rotate both q and k by the same extra offset and the dot product is numerically unchanged to machine precision.
- Side benefit: rotation is ORTHOGONAL ⇒ norm-preserving. `||RoPE(x)|| = ||x||`, so RoPE doesn't distort the magnitude the downstream softmax sees — unlike sinusoidal which ADDS position to the embedding and inflates norms.
- Cost: 0 learnable parameters, d/2 (cos, sin) pairs precomputed once per (seq_len, d_head) at startup, O(B·T·H·D) elementwise multiplies per attention call.
</details>

> ⚠️ **Trap:** Claiming RoPE is 'sinusoidal inside attention'. It's a MULTIPLICATIVE rotation, not an additive sin/cos shift — and the inner product (not the embedding) is what encodes relative position.
>
> 📚 **References:** https://arxiv.org/abs/2104.09864, https://blog.eleuther.ai/rotary-embeddings/

---

### 🧑‍💻 Whiteboard Challenge: RoPE from scratch — verify inner-product relative-distance property

**Prompt:** Implement `apply_rope(x, pos, theta_base=10000.0)` that takes a (..., d_head) tensor `x` and a scalar `pos`, and returns the position-m-rotated vector. Then verify the RELATIVE-DISTANCE PROPERTY numerically: for any offset Δ, `<RoPE(q, m), RoPE(k, n)>` must equal `<RoPE(q, m+Δ), RoPE(k, n+Δ)>` to machine precision.

**Constraints:**
- Use the real-valued form: split the last dim into d_head/2 pairs, rotate each pair by `m · θ_j` where `θ_j = theta_base^(-2j/d_head)`.
- Use MLX throughout — no numpy or torch. The final dot product must go through `mx.sum` and `mx.eval`.
- Generate q and k with `mx.random.normal` (seed the PRNG for reproducibility) at (d_head=128).
- Assert that `<RoPE(q, m), RoPE(k, n)>` matches `<RoPE(q, m+Δ), RoPE(k, n+Δ)>` for Δ ∈ {1, 5, 37} within abs tolerance 1e-3 (float32; rotations accumulate error).
- Include one additional `assert` that `||RoPE(q, m)||_2 == ||q||_2` to within 1e-4 — the orthogonality / norm-preservation check.

**Expected complexity:** Precompute: O(T · d_head) for the (cos, sin) tables. Apply: O(B · T · H · d_head) elementwise multiplies per attention call.

In [12]:
import math
import mlx.core as mx

def rope_freqs(d_head: int, max_pos: int, theta_base: float = 10000.0):
    """Precompute (cos, sin) tables of shape (max_pos, d_head/2)."""
    j = mx.arange(0, d_head, 2, dtype=mx.float32)  # (d_head/2,)
    thetas = 1.0 / (theta_base ** (j / d_head))    # (d_head/2,)
    positions = mx.arange(0, max_pos, dtype=mx.float32)  # (max_pos,)
    # Outer product: angles[m, j] = m * theta_j
    angles = positions[:, None] * thetas[None, :]  # (max_pos, d_head/2)
    return mx.cos(angles), mx.sin(angles)

def apply_rope(x: mx.array, pos: int, cos_tab: mx.array, sin_tab: mx.array) -> mx.array:
    """Rotate x (shape (..., d_head)) by RoPE at absolute position pos.

    Pair even/odd dims: (x0, x1, x2, x3, ...) -> pairs (x0,x1), (x2,x3), ...
    Each pair gets rotated by angle m*theta_j via a 2x2 rotation matrix.
    """
    c = cos_tab[pos]  # (d_head/2,)
    s = sin_tab[pos]  # (d_head/2,)
    # Split last dim into even/odd.
    x_even = x[..., 0::2]  # (..., d_head/2)
    x_odd = x[..., 1::2]   # (..., d_head/2)
    # 2x2 rotation applied to each pair.
    rot_even = x_even * c - x_odd * s
    rot_odd = x_even * s + x_odd * c
    # Interleave even/odd back to original layout.
    out = mx.stack([rot_even, rot_odd], axis=-1)  # (..., d_head/2, 2)
    return out.reshape(x.shape)

# Setup: d_head=128, seed for determinism.
d_head = 128
max_pos = 128
mx.random.seed(0)
q = mx.random.normal(shape=(d_head,))
k = mx.random.normal(shape=(d_head,))
cos_tab, sin_tab = rope_freqs(d_head, max_pos, theta_base=10000.0)
mx.eval(cos_tab, sin_tab, q, k)

# --- Property 1: RoPE preserves norms (rotation is orthogonal). ---
m = 7
q_rot = apply_rope(q, m, cos_tab, sin_tab)
norm_before = float(mx.sqrt(mx.sum(q * q)).item())
norm_after = float(mx.sqrt(mx.sum(q_rot * q_rot)).item())
assert abs(norm_before - norm_after) < 1e-4, (
    f"norm not preserved: {norm_before:.6f} vs {norm_after:.6f}"
)

# --- Property 2: inner product depends only on (m - n). ---
# Pick base positions (m, n) and assert shifting both by Delta is invariant.
m, n = 10, 25  # m - n = -15
q_m = apply_rope(q, m, cos_tab, sin_tab)
k_n = apply_rope(k, n, cos_tab, sin_tab)
ref = float(mx.sum(q_m * k_n).item())
for delta in (1, 5, 37):
    q_md = apply_rope(q, m + delta, cos_tab, sin_tab)
    k_nd = apply_rope(k, n + delta, cos_tab, sin_tab)
    shifted = float(mx.sum(q_md * k_nd).item())
    assert abs(ref - shifted) < 1e-3, (
        f"relative-distance property violated at Δ={delta}: "
        f"{ref:.6f} vs {shifted:.6f}"
    )

# Force eval and print a summary line.
summary = mx.array([ref, norm_before, norm_after], dtype=mx.float32)
mx.eval(summary)
print(f"✅ norm preserved: {norm_before:.4f} -> {norm_after:.4f}")
print(f"✅ <RoPE(q,{m}), RoPE(k,{n})> = {ref:.4f}")
print(f"✅ invariant to joint shift by Δ ∈ {{1, 5, 37}}")


✅ norm preserved: 12.3879 -> 12.3879
✅ <RoPE(q,10), RoPE(k,25)> = 19.9685
✅ invariant to joint shift by Δ ∈ {1, 5, 37}


---

### 📐 Complexity & Systems: RoPE application per attention op — (B·T·H·D) elementwise

| Quantity | Formula / Value | Notes |
|---|---|---|
| FLOPs | `O(B · T · H · d_head) per attention call — 4 fused multiply-adds per pair per token per head. At B=4, T=2048, H=32, d_head=128 that's ~67 M FLOPs, negligible against the O(B·T²·H·d_head) attention matmul itself` | per forward pass |
| Memory | `Precomputed (cos, sin) tables: O(T · d_head) floats — shared ACROSS all (B, H) at inference time. At T=128k, d_head=128 that's ~33 MiB fp32 (or 16 MiB fp16). Typically cached in a module-level buffer and NEVER recomputed` | working set |
| Latency (M4 Pro, MLX) | `RoPE apply on M4 Pro for (B=2, T=2048, H=32, d_head=128) bf16: ~0.2–0.5 ms/call — well below 1% of end-to-end decoder layer time. Measured below` | measured, see paired benchmark cell |

💡 **Scaling implication:** RoPE is a tiny fraction of attention cost but the (cos, sin) table GROWS LINEARLY with max_seq_len. At 1M context and d_head=128 the cache is 256 MiB fp32 — still dominated by KV cache but meaningful. Frontier servers (vLLM, SGLang) share ONE (cos, sin) table across all requests.

In [13]:
# Benchmark: RoPE application on (B, T, H, d_head) bf16 tensors
import time
import math
import mlx.core as mx

def _rope_freqs(d_head: int, max_pos: int, theta_base: float = 10000.0):
    j = mx.arange(0, d_head, 2, dtype=mx.float32)
    thetas = 1.0 / (theta_base ** (j / d_head))
    positions = mx.arange(0, max_pos, dtype=mx.float32)
    angles = positions[:, None] * thetas[None, :]
    return mx.cos(angles), mx.sin(angles)

def apply_rope_batched(x: mx.array, cos_tab: mx.array, sin_tab: mx.array) -> mx.array:
    """Vectorized RoPE: x is (B, T, H, d_head); cos/sin are (T, d_head/2)."""
    # Broadcast (T, d/2) -> (1, T, 1, d/2) to match x's (B, T, H, d_head/2).
    c = cos_tab[None, :, None, :]
    s = sin_tab[None, :, None, :]
    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]
    rot_even = x_even * c - x_odd * s
    rot_odd = x_even * s + x_odd * c
    out = mx.stack([rot_even, rot_odd], axis=-1)
    return out.reshape(x.shape)

# Production-like shape: B=2, T=2048, H=32, d_head=128.
B, T, H, d_head = 2, 2048, 32, 128
mx.random.seed(0)
x = mx.random.normal(shape=(B, T, H, d_head)).astype(mx.bfloat16)
cos_tab, sin_tab = _rope_freqs(d_head, T, 10000.0)
cos_tab = cos_tab.astype(mx.bfloat16)
sin_tab = sin_tab.astype(mx.bfloat16)
mx.eval(x, cos_tab, sin_tab)

# Warmup (Requirement 5.3).
for _ in range(3):
    y = apply_rope_batched(x, cos_tab, sin_tab)
    mx.eval(y)

N = 20
t0 = time.perf_counter()
for _ in range(N):
    y = apply_rope_batched(x, cos_tab, sin_tab)
    mx.eval(y)
dt_ms = (time.perf_counter() - t0) / N * 1000.0

# Analytic FLOPs: 4 fma per pair × (d_head/2 pairs) × B × T × H
# Report rough 'FLOP rate' just so readers see how fast this is.
flops_per_call = 4 * (d_head // 2) * B * T * H
gflops = flops_per_call / (dt_ms * 1e-3) / 1e9

# Invariant: norm preservation across the whole batch.
norm_x = mx.sqrt(mx.sum(x.astype(mx.float32) ** 2, axis=-1))
norm_y = mx.sqrt(mx.sum(y.astype(mx.float32) ** 2, axis=-1))
diff = float(mx.max(mx.abs(norm_x - norm_y)).item())
# bf16 rotation introduces a few parts in 1e-2; loosen to 5e-2.
assert diff < 5e-2, f"RoPE norm drift {diff:.4f} > 5e-2 (bf16 rounding)"

print(f"shape: B={B} T={T} H={H} d_head={d_head}  (bf16)")
print(f"RoPE apply: {dt_ms:7.3f} ms / call   ({gflops:6.1f} GFLOP/s)")
print(f"max |norm(x) - norm(RoPE(x))| = {diff:.4e}  (bf16; rotation ≈ orthogonal)")


shape: B=2 T=2048 H=32 d_head=128  (bf16)
RoPE apply:   2.520 ms / call   (  13.3 GFLOP/s)
max |norm(x) - norm(RoPE(x))| = 2.2511e-02  (bf16; rotation ≈ orthogonal)


---
## Part 4: Comparison — Sinusoidal vs RoPE vs Learned Embeddings

| Property | Sinusoidal (2017) | Learned Embeddings | RoPE (2021) |
|---|---|---|---|
| **How it works** | Add fixed sin/cos to embeddings | Add learned vectors (one per position) | Rotate Q, K by position-dependent angle |
| **Parameters** | 0 (fixed) | `max_seq_len × d_model` | 0 (fixed) |
| **Position type** | Absolute | Absolute | **Relative** (via rotation) |
| **Extrapolation** | Moderate (smooth waves) | ❌ None (unseen positions = random) | ✅ Good (rotation generalizes) |
| **Applied to** | Embeddings (before attention) | Embeddings (before attention) | Q and K only (inside attention) |
| **Norm preservation** | ❌ No (addition changes norm) | ❌ No (addition changes norm) | ✅ Yes (rotation is orthogonal) |
| **Used in** | Original Transformer, BERT | GPT-2, early GPT-3 | **LLaMA, Gemma, Mistral, Qwen, Phi** |

### Why RoPE Wins for Modern LLMs

💡 **Relative position**: Attention scores `qᵀk` naturally encode the *distance* between tokens, not their absolute positions. This is more linguistically meaningful — "the word 3 tokens ago" matters more than "the word at position 47."

⚡ **Length extrapolation**: RoPE can handle sequences longer than training length because rotation angles are continuous. Combined with techniques like NTK-aware scaling and YaRN, RoPE-based models can extend from 4K to 128K+ context.

🎯 **Interview tip**: If asked "why not just learn position embeddings?" — learned embeddings can’t generalize beyond `max_seq_len` seen during training, and they add parameters. RoPE is parameter-free, norm-preserving, and encodes relative position — three wins in one.

In [14]:
# Quick comparison: parameter counts
vocab_size = 32000  # typical LLM vocab
d_model = 4096      # typical LLM hidden size
max_seq = 8192      # typical context length

learned_params = max_seq * d_model
sinusoidal_params = 0
rope_params = 0

print("Positional Encoding Parameter Comparison")
print("=" * 50)
print(f"Model config: d_model={d_model}, max_seq={max_seq}")
print(f"")
print(f"  Sinusoidal:  {sinusoidal_params:>12,} params (fixed, no learning)")
print(f"  RoPE:        {rope_params:>12,} params (fixed, no learning)")
print(f"  Learned:     {learned_params:>12,} params ({learned_params * 2 / 1e6:.1f} MB in float16)")
print(f"")
print(f"💡 For a 7B parameter model, learned PE adds {learned_params/7e9*100:.2f}% overhead.")
print(f"   Not huge, but RoPE gives better extrapolation for free.")
print()
print("Modern LLM Positional Encoding Choices:")
print("-" * 50)
for name, pe_type in [
    ("GPT-2 / GPT-3", "Learned absolute"),
    ("Original Transformer", "Sinusoidal"),
    ("LLaMA 1/2/3", "RoPE"),
    ("Gemma 1/2", "RoPE"),
    ("Mistral / Mixtral", "RoPE"),
    ("Qwen 2.5", "RoPE"),
    ("Phi-3", "RoPE"),
]:
    print(f"  {name:<25} → {pe_type}")

Positional Encoding Parameter Comparison
Model config: d_model=4096, max_seq=8192

  Sinusoidal:             0 params (fixed, no learning)
  RoPE:                   0 params (fixed, no learning)
  Learned:       33,554,432 params (67.1 MB in float16)

💡 For a 7B parameter model, learned PE adds 0.48% overhead.
   Not huge, but RoPE gives better extrapolation for free.

Modern LLM Positional Encoding Choices:
--------------------------------------------------
  GPT-2 / GPT-3             → Learned absolute
  Original Transformer      → Sinusoidal
  LLaMA 1/2/3               → RoPE
  Gemma 1/2                 → RoPE
  Mistral / Mixtral         → RoPE
  Qwen 2.5                  → RoPE
  Phi-3                     → RoPE


---
## Summary & Key Takeaways

**What we built in this notebook:**
1. **Token embeddings** — a lookup table mapping token IDs to dense vectors (`nn.Embedding`)
2. **Sinusoidal PE** — fixed sin/cos waves that give each position a unique fingerprint
3. **RoPE** — rotation-based encoding that preserves norms and encodes relative position

**Key mental models:**
- 💡 Transformers are permutation-invariant — position must be explicitly injected
- 💡 RoPE encodes position through *rotation*, not addition — this preserves norms and enables relative position attention
- ⚡ RoPE is parameter-free and applied only to Q, K — not V
- 🎯 Modern LLMs (LLaMA, Gemma, Mistral) all use RoPE because it extrapolates to longer sequences
- ⚠️ Learned position embeddings can't generalize beyond training length

**Next up:** [Notebook 05: Self-Attention from Scratch](05_self_attention.ipynb) — where we use these embeddings as input to the attention mechanism.

## 📜 History & Alternatives

### The Evolution of Positional Encoding

Transformers have no inherent notion of token order — positional encodings inject sequence position information. The field evolved from fixed mathematical functions to learned and relative schemes that enable length generalization.

| Year | Method | Team | Key Contribution |
|------|--------|------|-----------------|
| 2017 | **Sinusoidal PE** | Vaswani et al. (Google) | Fixed sin/cos functions at different frequencies — original Transformer |
| 2017 | **Learned PE** | Vaswani et al. (Google) | Trainable position embeddings — used in GPT-1, BERT |
| 2018 | **Relative PE** | Shaw et al. (Google) | Encode relative distance between tokens, not absolute position |
| 2021 | **Rotary PE (RoPE)** | Su et al. (Zhuiyi Tech) | Rotate Q/K vectors by position-dependent angles — encodes relative position through dot products |
| 2021 | **ALiBi** | Press et al. (Meta) | No positional embedding — add linear bias to attention scores based on distance |
| 2022 | **xPos** | Sun et al. | Extended RoPE with exponential decay for better length extrapolation |
| 2023 | **YaRN** | Peng et al. | RoPE scaling via NTK-aware interpolation — extends context without fine-tuning |
| 2023 | **LongRoPE** | Microsoft | Progressive interpolation for 2M+ context lengths |
| 2024 | **p-RoPE (partial RoPE)** | Google DeepMind (Gemma 2/3) | Apply RoPE to only a fraction of head dimensions — rest are position-free |

### 💡 Why RoPE Won

RoPE became the dominant positional encoding (used in LLaMA, Mistral, Gemma, Qwen, and most modern LLMs) because of three properties:

1. **Relative position via dot product**: `⟨RoPE(q, m), RoPE(k, n)⟩` depends only on `m - n`, not absolute positions
2. **Decaying with distance**: Attention naturally decreases for distant tokens (desirable inductive bias)
3. **Length extrapolation**: With interpolation tricks (YaRN, NTK-aware), RoPE models can generalize to longer sequences than seen during training

### Positional Encoding Comparison

| Method | Type | Length Generalization | Used By | Complexity |
|--------|------|---------------------|---------|------------|
| Sinusoidal | Absolute, fixed | Poor beyond training length | Original Transformer | O(1) per position |
| Learned | Absolute, trained | Poor — hard limit at max position | GPT-2, BERT | O(1) per position |
| RoPE | Relative, fixed | Good with interpolation | LLaMA, Mistral, Qwen | O(d) rotation per head |
| ALiBi | Relative, fixed | Excellent — no position limit | BLOOM, MPT | O(1) bias per attention pair |
| p-RoPE | Partial relative | Excellent | Gemma 2, Gemma 3, Gemma 4 | O(d/2) rotation on subset |

### ⚡ The Context Length Revolution

The evolution of positional encodings directly enabled the context length explosion:
- 2017: 512 tokens (original Transformer)
- 2020: 2,048 tokens (GPT-3)
- 2023: 8,192 → 32,768 tokens (LLaMA 2 → Mistral)
- 2024: 128K → 1M tokens (GPT-4 Turbo → Gemini 1.5)
- 2025: 2M+ tokens (Gemini 2.5)

Each jump required better positional encodings (RoPE interpolation, ALiBi) combined with efficient attention (Flash Attention, Ring Attention).

### ⚠️ Common Pitfalls

1. **Forgetting to scale RoPE frequencies when extending context**: Naively using a model beyond its trained context length causes catastrophic attention score degradation. You must apply NTK-aware interpolation or YaRN scaling — never just increase `max_seq_len` without adjusting the frequency base.
2. **Mixing absolute and relative PE assumptions**: Some architectures (e.g., BERT) bake absolute position IDs into the model. Switching to RoPE requires removing the old position embeddings entirely, not layering them.
3. **Ignoring the base frequency (`theta`)**: RoPE's `theta` parameter (default 10,000) controls the wavelength spectrum. Models trained with one `theta` value perform poorly if you change it at inference without retraining or careful interpolation.

### 🎯 Interview Tip

> *"RoPE encodes position by rotating query and key vectors in 2D subspaces — the rotation angle is proportional to position × frequency. When you compute the dot product Q·K, the rotation angles subtract, making the result depend only on relative position (m-n). This is mathematically elegant because it achieves relative positional encoding without modifying the attention mechanism itself — you just rotate the inputs. The partial-RoPE variant in Gemma applies rotation to only some head dimensions, leaving others position-free to capture global (position-independent) patterns."*

---

### 🎯 Interview Question nb04-q03  ·  [core]  ·  research_engineer, systems_engineer

**Q:** Explain ALiBi. Write the slope formula for H heads, reason through why it's specifically `2^(-8/H · head_idx)` (a geometric series 1/2^i), and explain why this particular choice enables length extrapolation beyond training context.

<details>
<summary>Key points in a strong answer</summary>

- ALiBi (Attention with Linear Biases, Press et al. 2021): NO positional embedding is added to Q/K. Instead, before the softmax, add a linear bias to the (m, n) attention logit: `A[m,n] ← Q[m]·K[n] − m_h · |m − n|`, where `m_h` is a per-HEAD negative slope.
- Slope formula for H heads: `m_h = 2^(-8 · h / H)` for head index h ∈ {1, ..., H}. That's a geometric series: slopes for H=8 are 1/2, 1/4, 1/8, ..., 1/256 — spanning 8 octaves of distance-decay rates.
- Why 2^(-8/H): the 2^(-8) = 1/256 is the smallest slope in the series; it's chosen so the LOWEST-slope head attends roughly uniformly across 256-ish tokens (log2(256)=8) — large-context semantic head. The HIGHEST-slope head (h=1, slope ≈ 1/2) attends almost exclusively to the immediately preceding token — local-context syntactic head.
- The geometric spacing is deliberate: a LINEAR spread of slopes would cluster all heads near one decay rate. Geometric covers the full 1×..256× distance range with only H heads — classic multi-scale decomposition.
- Length extrapolation: ALiBi is a PURE FUNCTION of `|m − n|`. Training on sequences of length L and evaluating at 4L still uses the same slopes on the same distance metric — no 'out-of-distribution positions' exist. The 2021 paper demonstrates ALiBi-512 extrapolates cleanly to 16k without retraining.
- Trade-off: ALiBi REGRESSES to distance-weighted attention at long range — the bias dominates Q·K for far-apart tokens. Great for long-document tasks that actually have locality; poor for tasks needing long-distance reasoning. This is why LLaMA-1/2/3 picked RoPE (+ YaRN scaling) over ALiBi.
- Used by: BLOOM-176B (2022), MPT-7B (2023), Falcon-RW. Discontinued at the frontier once RoPE + NTK/YaRN matched or beat ALiBi's extrapolation while preserving uniform attention at distance.
</details>

> ⚠️ **Trap:** Writing the slope as linear in h (slope ∝ h/H) — it's EXPONENTIAL: `slope = 2^(-8·h/H)`. Linear slopes collapse all heads onto a narrow decay band and destroy the multi-scale property.
>
> 📚 **References:** https://arxiv.org/abs/2108.12409, https://github.com/ofirpress/attention_with_linear_biases

---

### 🎯 Interview Question nb04-q04  ·  [core]  ·  research_engineer

**Q:** What is NoPE (2023) and what's the surprising finding? If transformers are permutation-invariant, how can a model with ZERO positional encoding learn any order-sensitive task at all?

<details>
<summary>Key points in a strong answer</summary>

- NoPE (Kazemnejad et al., 'The Impact of Positional Encoding on Length Generalization', NeurIPS 2023): strip ALL positional encoding — no sinusoidal, no learned PE, no RoPE, no ALiBi — and train a DECODER-ONLY (causal) transformer.
- The surprising finding: NoPE decoder-only models match or BEAT RoPE / ALiBi on several length-generalization benchmarks. They extrapolate from 512 → 2048 context cleanly, while learned-PE models break immediately and RoPE without YaRN degrades past training length.
- How it's possible: decoder-only transformers are NOT permutation-invariant — the CAUSAL MASK breaks the symmetry. Position n's representation attends to positions {0, ..., n} only; position 0 attends only to {0}. Each layer computes a DIFFERENT function at each position purely because the set of visible tokens differs.
- Concretely, the first-layer attention output at position n is Σ_{i ≤ n} softmax(q_n·k_i)·v_i — it implicitly encodes 'I am at least position n' (because n+1 tokens went into the sum) without any explicit position signal.
- Caveat — this is NOT true for encoder-only / bidirectional models. Remove PE from BERT and every position sees identical context; the model is genuinely permutation-invariant and cannot learn 'The cat sat on the mat' vs 'mat the on sat cat The'.
- Practical significance: the research frontier is exploring hybrid stacks — NoPE layers interleaved with RoPE layers (Llama 4 preview, some DeepSeek ablations) to get the length-generalization of NoPE AND the precise relative-position control of RoPE.
- When NoPE wins in practice: tasks where the causal-mask ordering signal is already sufficient (copy/retrieval, count, pattern continuation). When it loses: tasks needing precise relative-offset matching (e.g. 'what's the 17th word from the end').
</details>

> ⚠️ **Trap:** Asserting 'NoPE works because transformers don't need position info' — they DO; the causal mask SUPPLIES the ordering signal. Remove the mask (encoder-only) and NoPE breaks completely.
>
> 📚 **References:** https://arxiv.org/abs/2305.19466, https://openreview.net/forum?id=wnuCFuKeYl

---

### 🎯 Interview Question nb04-q05  ·  [stretch]  ·  research_engineer, systems_engineer

**Q:** A model was trained with RoPE at 4K context. You want to extend to 128K at inference time with MINIMAL fine-tuning. Walk through position interpolation (PI), NTK-aware scaling, and YaRN — what each changes, and why YaRN is the current production default.

<details>
<summary>Key points in a strong answer</summary>

- Problem: at test-time position 128000 with a model trained to max position 4096, the RoPE angle `m·θ_j` is 32× larger than anything seen during training. High-frequency dimensions wrap around aggressively; attention scores collapse to noise.
- Position Interpolation (PI, Chen et al. 2023): LINEARLY rescale all positions before applying RoPE: `m' = m · L_train / L_test = m / s` where s = 32 for 4K→128K. Every dimension is squeezed by the SAME factor. Requires a small fine-tune (~1B tokens) to recover quality. Simple; works; squeezes all frequencies equally.
- NTK-aware scaling (bloc97, June 2023): different dimensions get different scaling. HIGH-frequency dims (small θ_j) are NOT rescaled — they already wrap many times per 4K so one more wrap is fine. LOW-frequency dims (large θ_j) ARE rescaled — they barely finish one cycle in 4K and must be stretched. Achieved by rescaling `base = 10000 → 10000 · s^(d/(d−2))` rather than the positions themselves. Training-free on short contexts; degrades on very long ones.
- YaRN (Peng et al. 2023, 'Yet another RoPE extensioN'): combines NTK-aware scaling with an explicit ATTENTION TEMPERATURE correction. The observation: as you stretch RoPE, the effective attention-score distribution cools (std shrinks) because close-together positions now look MORE similar. YaRN multiplies logits by `sqrt(1 + 0.1·log(s))` to restore the original temperature.
- YaRN also uses a RAMP function between NTK-style and PI-style scaling: low-freq dims get full PI (preserves smooth extrapolation), high-freq dims get 1× (preserves local precision), mid-freq dims get a smooth interpolation. Three regimes, one hyperparam.
- Production defaults (2024–2025): LLaMA-3.1 uses a YaRN variant at 128K; LLaMA-3.2 ships 128K out-of-the-box with a rescaled RoPE base of 500_000 (!) in addition to YaRN. DeepSeek-V3 extends to 128K with YaRN. Qwen-2.5 uses YaRN + Dual-Chunk Attention for 1M context.
- Fine-tuning budget: PI → ~1B tokens to recover, NTK-aware → ~100M tokens, YaRN → often zero-shot competitive, ~50M tokens to recover completely. That's the main reason YaRN is production default.
</details>

> ⚠️ **Trap:** Treating NTK-aware scaling as position rescaling. It's a BASE (θ_base) rescaling — positions are left alone, the RoPE frequency schedule is compressed. Confusing these two produces subtly wrong interpolation code that CLIP-degrades attention at long range.
>
> 📚 **References:** https://arxiv.org/abs/2306.15595, https://arxiv.org/abs/2309.00071, https://www.reddit.com/r/LocalLLaMA/comments/14lz7j5/ntkaware_scaled_rope_allows_llama_models_to_have/

---

### 🎯 Interview Question nb04-q06  ·  [stretch]  ·  mle, systems_engineer

**Q:** Explain the `rope_base` (θ_base, usually 10000) hyperparameter in RoPE. Why did LLaMA-1 use 10000 but LLaMA-3.1 use 500000? What exactly breaks if you get this wrong at inference time?

<details>
<summary>Key points in a strong answer</summary>

- RoPE's per-dimension frequencies are `θ_j = θ_base^(-2j/d)` for j ∈ {0, ..., d/2 − 1}. At `θ_base = 10000, d = 128`: j=0 rotates at freq 1.0 (period 2π), j=63 at freq 10000^(-1) ≈ 1e-4 (period 2π · 10000 ≈ 62800 tokens).
- θ_base controls the WAVELENGTH RANGE of position-dependent signals. Larger θ_base ⇒ the lowest-frequency dim has a LONGER period ⇒ the model can resolve position differences over a longer absolute span without aliasing.
- LLaMA-1 (4K training context): θ_base = 10000. Longest period ≈ 62800 tokens — comfortable headroom above 4K.
- LLaMA-3.1 (128K training context): θ_base = 500000. Longest period ≈ 3.1M tokens. With θ_base=10000 and 128K context, the lowest-freq dim wraps through nearly 2 full cycles — the model would alias position 0 with position 62800.
- Rule of thumb: `θ_base ≈ 100 · L_train` keeps the lowest-freq period safely above training context. Meta's 500_000 value for 128K corresponds roughly to that rule (500000 / 128000 ≈ 3.9× margin).
- Serving-time bug: load LLaMA-3.1 weights but use the config's `rope_theta=10000` (default for legacy models) and you get catastrophic long-context degradation — attention scores beyond ~20k are random. Diagnostic: perplexity curve by context length plateaus then EXPLODES past a specific threshold (the aliasing point).
- Fine-tuning recipe for context extension: first INCREASE `θ_base` to match the new target length, then apply YaRN (or equivalent) as a second correction. Meta's LLaMA-3 training run did both — they bumped θ_base AND used YaRN-style frequency rescaling.
</details>

> ⚠️ **Trap:** Thinking θ_base is 'just a scale factor'. It's the base of an EXPONENTIAL frequency schedule — doubling θ_base quadruples (not doubles) the lowest-dim period. Arithmetic mistakes here produce silent quality regressions in long-context serving.
>
> 📚 **References:** https://arxiv.org/abs/2309.16039, https://ai.meta.com/blog/meta-llama-3-1/, https://huggingface.co/blog/rope-scaling

---

### 🎯 Interview Question nb04-q07  ·  [research]  ·  research_engineer, systems_engineer

**Q:** In Grouped-Query Attention (LLaMA-2/3, Mistral, Qwen) a single K/V head is shared across G query heads. Walk through how RoPE interacts with GQA: which tensor gets rotated, is the rotation applied ONCE or G times per layer, and what's the production trade-off between caching PRE-rope vs POST-rope keys?

<details>
<summary>Key points in a strong answer</summary>

- GQA shape: `n_heads` query heads, `n_kv_heads = n_heads / G` K/V heads (G=8 for LLaMA-3-70B: 64 Q heads, 8 KV heads). RoPE is applied to BOTH Q and K — but K has fewer heads, so the RoPE cost on K is 1/G of the Q cost.
- Rotation is applied ONCE per K head (not G times). The SAME rotated K head is reused by all G query heads in its group via broadcasting at the score-matmul (or via `mx.repeat`/`einsum` in naive impls). Getting this wrong — re-rotating K once per query head — inflates RoPE cost by G and is a common first-GQA-bug.
- V is NEVER rotated. RoPE rotates the dot-product space (Q·K), not the value space. Value heads are shared across the query group without any position-dependent transform.
- Cache layout choice 1 — POST-rope KV cache (vLLM, SGLang, MLX-LM default): store `K_rotated = RoPE(K, pos)` in the cache. Pro: zero compute per decode step; one lookup, one matmul. Con: the cached keys are TIED to the position at which they were rotated — any context-extension retrofit (YaRN at serving time) requires invalidating the whole cache.
- Cache layout choice 2 — PRE-rope KV cache: store raw K, apply RoPE on-the-fly at attention time using the position index from the cache slot. Pro: trivially supports dynamic RoPE-base / YaRN swap at inference. Con: extra RoPE multiply per decoded token per K head (small — ~1% of attention cost).
- Production choice (2024-2025): frontier servers (vLLM, SGLang, TRT-LLM) default to POST-rope caching because the performance win at 128k context is material and YaRN-at-serving is rare. Research frameworks and some experimentation servers default to pre-rope so you can A/B test scaling factors without rebuilding the cache.
- Subtle bug: SGLang's RadixAttention prefix cache hashes (token_ids) of cached prefix chunks. If a user shares a prefix across two requests at DIFFERENT starting positions (e.g. chat session resuming mid-context), the POST-rope cache entries are VALID ONLY at their original positions — you cannot reuse them at shifted positions. SGLang detects this via a position-equality check on cache hit; a broken check silently corrupts generation.
</details>

> ⚠️ **Trap:** Answering 'RoPE is applied per attention head' and forgetting that under GQA only the K/V heads are rotated, not each query head. The Q-side rotation IS per-query-head (there's no sharing on Q), but the K-side is per-group — a factor of G difference in compute.
>
> 📚 **References:** https://arxiv.org/abs/2305.13245, https://arxiv.org/abs/2104.09864, https://github.com/sgl-project/sglang

---

### 🧑‍💻 Whiteboard Challenge: ALiBi slopes for H heads — verify the geometric progression

**Prompt:** Implement `alibi_slopes(n_heads)` returning an `mx.array` of H per-head slopes following the ALiBi paper's formula `m_h = 2^(-8 · h / H)` for h ∈ {1, ..., H}. Assert the result is strictly monotonically decreasing, lies in (0, 1], and that successive slopes form a GEOMETRIC progression with common ratio `2^(-8/H)`.

**Constraints:**
- Use MLX throughout — no numpy. The returned slopes array must be materialized via `mx.eval`.
- Support H = 8, 16, 32, 64 (powers of 2) and at least one non-power-of-two H = 12 (GPT-Neo uses 12 heads). The original paper's slope formula works for any H.
- Assert monotonicity: for all i, `slopes[i] > slopes[i+1]`.
- Assert geometric: `slopes[i+1] / slopes[i]` is within 1e-5 of `2^(-8/H)` for every adjacent pair.
- Assert the LARGEST slope (h=1) is `2^(-8/H)` and the SMALLEST (h=H) is `2^(-8) = 1/256` — the canonical endpoints.

**Expected complexity:** O(H) — trivially vectorized in a single MLX op.

In [15]:
import math
import mlx.core as mx

def alibi_slopes(n_heads: int) -> mx.array:
    """ALiBi slopes: m_h = 2^(-8 * h / H) for h in {1, ..., H}.

    Returns an fp32 array of shape (H,) in descending order of     magnitude (so the first head has the LARGEST decay, the     last has the SMALLEST).
    """
    if n_heads <= 0:
        raise ValueError(f"n_heads must be positive, got {n_heads}")
    h = mx.arange(1, n_heads + 1, dtype=mx.float32)  # (H,)
    return mx.power(2.0, -8.0 * h / n_heads)

# Check the canonical H values.
for H in (8, 16, 32, 64, 12):
    slopes = alibi_slopes(H)
    mx.eval(slopes)

    # (a) Range: all in (0, 1]. Largest is 2^(-8/H), smallest is 2^(-8)=1/256.
    slopes_list = slopes.tolist()
    assert slopes_list[0] <= 1.0 and slopes_list[-1] > 0.0, (
        f"H={H}: out-of-range slopes {slopes_list[0]}, {slopes_list[-1]}"
    )
    expected_max = 2.0 ** (-8.0 / H)
    expected_min = 2.0 ** -8.0
    assert abs(slopes_list[0] - expected_max) < 1e-5, (
        f"H={H}: max slope {slopes_list[0]} != {expected_max}"
    )
    assert abs(slopes_list[-1] - expected_min) < 1e-5, (
        f"H={H}: min slope {slopes_list[-1]} != {expected_min}"
    )

    # (b) Monotonic strictly decreasing.
    for i in range(H - 1):
        assert slopes_list[i] > slopes_list[i + 1], (
            f"H={H}: slopes not monotone at i={i}"
        )

    # (c) Geometric progression: ratio = 2^(-8/H).
    expected_ratio = 2.0 ** (-8.0 / H)
    for i in range(H - 1):
        ratio = slopes_list[i + 1] / slopes_list[i]
        assert abs(ratio - expected_ratio) < 1e-5, (
            f"H={H}: non-geometric at i={i} (ratio {ratio} != {expected_ratio})"
        )

    print(f"H={H:>3}: slopes[0]={slopes_list[0]:.6f}, "
          f"slopes[-1]={slopes_list[-1]:.6f}, ratio={expected_ratio:.6f}")

# Demonstrate H=8 explicitly — this is what BLOOM / MPT actually use.
slopes_8 = alibi_slopes(8)
mx.eval(slopes_8)
print("\nBLOOM/MPT-style H=8 slopes:", [f"{s:.4f}" for s in slopes_8.tolist()])
print("✅ geometric with common ratio 2^(-1) = 0.5; spans 8 octaves of decay")


H=  8: slopes[0]=0.500000, slopes[-1]=0.003906, ratio=0.500000
H= 16: slopes[0]=0.707107, slopes[-1]=0.003906, ratio=0.707107
H= 32: slopes[0]=0.840896, slopes[-1]=0.003906, ratio=0.840896
H= 64: slopes[0]=0.917004, slopes[-1]=0.003906, ratio=0.917004
H= 12: slopes[0]=0.629961, slopes[-1]=0.003906, ratio=0.629961

BLOOM/MPT-style H=8 slopes: ['0.5000', '0.2500', '0.1250', '0.0625', '0.0312', '0.0156', '0.0078', '0.0039']
✅ geometric with common ratio 2^(-1) = 0.5; spans 8 octaves of decay


---

### 📐 Complexity & Systems: KV cache under RoPE — pre-rope vs post-rope storage (GQA aware)

| Quantity | Formula / Value | Notes |
|---|---|---|
| FLOPs | `Per DECODE step: post-rope cache → 0 extra FLOPs (one matmul against the cached K). Pre-rope cache → +O(n_kv_heads · d_head) elementwise multiplies per cached position that participates in the dot-product — at n_kv_heads=8, d_head=128, T=128k that's ~130 MFLOPs/step, about 1% of attention cost` | per forward pass |
| Memory | `Cache footprint is IDENTICAL for both layouts: `2 · L · B · n_kv_heads · T · d_head · bytes_per_elem` (2 = K + V). At L=32, B=1, n_kv_heads=8, T=128k, d_head=128 bf16: ~16 GiB per request. The RoPE (cos, sin) table is shared across all B, L and is negligible by comparison` | working set |
| Latency (M4 Pro, MLX) | `On M4 Pro with (B=1, T=2048, n_kv_heads=8, d_head=128) bf16, applying RoPE lazily per attention call costs ~0.05 ms — so the pre-rope layout adds <1% to attention latency in exchange for dynamic rope-base swap capability. Measured below` | measured, see paired benchmark cell |

💡 **Scaling implication:** Post-rope: cache grows with T but NEVER gets re-rotated; fastest at serve time but rope-base / scaling must be frozen at cache-fill. Pre-rope: same footprint, ~1% compute overhead, but supports YaRN-at-serving and run-time rope-base swap. Frontier servers (vLLM, SGLang) default post-rope; research and experimentation stacks default pre-rope. GQA matters: the cache is `n_kv_heads`-wide, NOT `n_heads`-wide — LLaMA-3-70B saves 8× on cache size vs vanilla MHA.

In [16]:
# Benchmark: pre-rope vs post-rope K-cache attention latency (GQA-shaped)
import time
import math
import mlx.core as mx

def _rope_freqs(d_head: int, max_pos: int, theta_base: float = 10000.0):
    j = mx.arange(0, d_head, 2, dtype=mx.float32)
    thetas = 1.0 / (theta_base ** (j / d_head))
    positions = mx.arange(0, max_pos, dtype=mx.float32)
    angles = positions[:, None] * thetas[None, :]
    return mx.cos(angles), mx.sin(angles)

def apply_rope_kv(x: mx.array, cos_tab: mx.array, sin_tab: mx.array) -> mx.array:
    """Rotate x (B, T, n_kv_heads, d_head) by RoPE using (T, d_head/2) tables."""
    c = cos_tab[None, :, None, :]
    s = sin_tab[None, :, None, :]
    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]
    rot_even = x_even * c - x_odd * s
    rot_odd = x_even * s + x_odd * c
    out = mx.stack([rot_even, rot_odd], axis=-1)
    return out.reshape(x.shape)

# LLaMA-3-70B-ish GQA shape: n_heads=64 Q heads, n_kv_heads=8 (G=8), d_head=128.
B, T, n_kv_heads, n_heads, d_head = 1, 2048, 8, 64, 128
G = n_heads // n_kv_heads
mx.random.seed(0)

# A 'K-cache' containing T past keys (already rotated for the post-rope path).
k_raw = mx.random.normal(shape=(B, T, n_kv_heads, d_head)).astype(mx.bfloat16)
cos_tab, sin_tab = _rope_freqs(d_head, T, 10000.0)
cos_tab = cos_tab.astype(mx.bfloat16)
sin_tab = sin_tab.astype(mx.bfloat16)
k_post = apply_rope_kv(k_raw, cos_tab, sin_tab)

# New query at a single step: (B, 1, n_heads, d_head). RoPE at pos=T.
q = mx.random.normal(shape=(B, 1, n_heads, d_head)).astype(mx.bfloat16)
# For measurement we RoPE the query at position 0 -> position T still costs the same.
q_rot = apply_rope_kv(q, cos_tab[:1], sin_tab[:1])
mx.eval(k_raw, k_post, q_rot, cos_tab, sin_tab)

def attn_post_rope():
    """Post-rope path: Q·K^T directly against cached rotated keys."""
    # Broadcast shared KV heads to G query heads via reshape + repeat.
    k = mx.repeat(k_post, G, axis=2)             # (B, T, n_heads, d_head)
    # (B, 1, n_heads, d_head) @ (B, T, n_heads, d_head)^T on last dim.
    scores = mx.sum(q_rot * k, axis=-1)          # (B, T, n_heads)   [analytic stub]
    return scores

def attn_pre_rope():
    """Pre-rope path: apply RoPE to raw cached keys on every attention call."""
    k = apply_rope_kv(k_raw, cos_tab, sin_tab)   # extra work: ~1% of attn
    k = mx.repeat(k, G, axis=2)
    scores = mx.sum(q_rot * k, axis=-1)
    return scores

# Warmup (Requirement 5.3).
for _ in range(3):
    _y = attn_post_rope(); mx.eval(_y)
    _y = attn_pre_rope();  mx.eval(_y)

N = 20
t0 = time.perf_counter()
for _ in range(N):
    y1 = attn_post_rope(); mx.eval(y1)
post_ms = (time.perf_counter() - t0) / N * 1000.0

t0 = time.perf_counter()
for _ in range(N):
    y2 = attn_pre_rope(); mx.eval(y2)
pre_ms = (time.perf_counter() - t0) / N * 1000.0

# Analytic cache footprint — identical for both layouts.
cache_bytes = 2 * B * T * n_kv_heads * d_head * 2  # 2 (K+V) * ... * 2 (bf16)
cache_mib = cache_bytes / (1024 * 1024)
overhead_pct = (pre_ms - post_ms) / post_ms * 100.0

# Invariants: both paths produce the SAME scores up to bf16 rounding.
diff = float(mx.max(mx.abs(y1.astype(mx.float32) - y2.astype(mx.float32))).item())
# bf16 non-associativity allows a few parts in 1e0 on (T=2048)-sum scores.
assert diff < 5.0, f"post-rope and pre-rope disagree by {diff:.4f}"

print(f"GQA shape: n_heads={n_heads}, n_kv_heads={n_kv_heads}, G={G}, T={T}, d_head={d_head}")
print(f"KV cache footprint (analytic, bf16): {cache_mib:.1f} MiB  (identical both layouts)")
print(f"post-rope attention:  {post_ms:6.3f} ms / call")
print(f"pre-rope  attention:  {pre_ms:6.3f} ms / call   (+{overhead_pct:+.1f}%)")
print(f"max |post - pre| scores diff: {diff:.4f}  (bf16 rounding only — same attention)")


GQA shape: n_heads=64, n_kv_heads=8, G=8, T=2048, d_head=128
KV cache footprint (analytic, bf16): 8.0 MiB  (identical both layouts)
post-rope attention:   0.808 ms / call
pre-rope  attention:   0.882 ms / call   (++9.2%)
max |post - pre| scores diff: 0.0000  (bf16 rounding only — same attention)


---

### 🏭 How Production Systems Handle This — RoPE caching & rope-base scaling in long-context servers

| System | Mechanism | Notes |
|---|---|---|
| vLLM | Precomputes (cos, sin) tables ONCE at model-load time for the full `max_model_len` and broadcasts them across all attention layers. For models with RoPE scaling (LLaMA-3.1, DeepSeek-V3) reads `rope_scaling` from `config.json` and builds a YaRN-/NTK-aware table; `rope_theta` (θ_base) is honored per-model. The table lives in GPU memory and is shared across concurrent requests — ~16 MiB fp16 at 128k. | |
| SGLang | Same RoPE precompute strategy as vLLM; additionally the RadixAttention prefix cache assumes RoPE has been applied BEFORE the KV entries are hashed — prefix cache hits therefore require identical `rope_theta` and scaling across a shared-prefix batch, or the cache is silently wrong. | |
| TensorRT-LLM | Emits RoPE as a fused CUDA kernel with the (cos, sin) tables uploaded as constant memory. Supports llama-style + GPTNeoX-style rotation conventions (they differ in whether pairs are (0,1),(2,3),... or (0,d/2),(1,d/2+1),... — mix them and you get silent garbage). | |
| MLX-LM | Uses `mlx.nn.RoPE` with per-model `rope_theta` pulled from the HuggingFace config; caches the (cos, sin) tables on CPU and uploads to GPU on first use. For LLaMA-3.1-style YaRN scaling, MLX-LM reads `rope_scaling.type` and dispatches to the appropriate frequency schedule (linear / dynamic-NTK / YaRN). The tables are shared-memory friendly thanks to UMA. | |

🎯 **Interview tip:** Know at least one concrete trade-off per row.

---

### 🔭 Frontier Context (Context extension via RoPE scaling (2024–2026))

**Paper trail:**
1. YaRN: Efficient Context Window Extension (Peng et al.) (2023) — Combines NTK-aware frequency rescaling with an explicit attention-temperature correction. Becomes the 2024 production default — LLaMA-3.1, DeepSeek-V3, Qwen-2.5 all ship YaRN variants.
2. LLaMA-3.1 Technical Report (Meta) (2024) — Native 128k context via YaRN + BUMPED θ_base (10000 → 500000). First frontier open-weights model with 100k+ context out-of-the-box; demonstrates that θ_base is as important as the scaling factor.
3. DeepSeek-V3 Technical Report (DeepSeek) (2024) — 128k context with YaRN; documents the 'two-stage' context extension recipe: pretrain short, extend with YaRN + ~50M-token continued pretraining at the target length. Reproducibility recipe for any open model.
4. Gemma-3 Technical Report (Google DeepMind) (2025) — Hybrid local/global attention with INTERLEAVED RoPE scaling: local layers use θ_base=10000 (short-range precision), global layers use θ_base=1_000_000 (128k context). Demonstrates that RoPE-base can vary per-layer — a 2025 generalization of LLaMA-3.1's single-θ recipe.
5. LongRoPE: Extending LLMs to 2M Context (Microsoft) (2024) — Progressive interpolation: find per-dimension frequency rescaling factors via a short non-linear search rather than analytically. Extends Mistral-7B to 2M tokens with ~1B tokens of fine-tuning; the frontier of RoPE-scaling research.

**Current SOTA:** As of late 2025 the frontier recipe is: (a) train short with a moderately large θ_base (LLaMA-3 uses 500000, Gemma-3 global layers use 1_000_000), then (b) apply YaRN to extend context 16–32× with ~50M tokens of continued pretraining. LLaMA-3.1 (128k), DeepSeek-V3 (128k), Qwen-2.5-1M (1M via YaRN + Dual-Chunk Attention), Gemma-3 (128k via hybrid local/global RoPE). Active frontiers: (i) per-layer θ_base (Gemma-3), (ii) learned frequency schedules (LongRoPE's non-analytic θ_j), (iii) hybrid NoPE/RoPE stacks.

---

### 🛠️ Failure Modes & Debugging: Long-context RoPE bugs — silent quality cliffs, off-by-one position indexing, and freq-base / precision mismatches

**Root causes (ranked by frequency):**
1. Off-by-one in RoPE position indexing: mixing up 0-indexed (first token is position 0) vs 1-indexed conventions, or confusing ABSOLUTE position (index in the full sequence) with CACHE-SLOT position (index in the KV cache) at decode time. Fix: every production path passes `past_len + i` for the i-th new token where `past_len` is the cached-prefix length; one-off the base and the relative distances (m − n) all shift by 1 and long-range attention degrades.
2. RoPE freq-base mismatch at serve time: the checkpoint was trained at `rope_theta = 500000` (LLaMA-3.1 convention) but the inference config loads with the legacy default `rope_theta = 10000` — attention wraps around at ~62k tokens and perplexity explodes past that point. Fix: ALWAYS read `rope_theta` from the model's `config.json`; never assume a default. Diagnostic: assert the runtime `rope_theta` exactly matches the checkpoint's value.
3. fp16 / bf16 precision loss in cos / sin at high positions: `m · θ_j` for small θ_j (late dims) grows linearly with m. Computing `cos(m · θ_j)` in bf16 at m > 65k loses all precision — `sin` wraps randomly. Fix: always precompute the (cos, sin) tables in fp32 and cast DOWN to bf16 only at use. Every production server does this.

**Diagnostic code below reproduces the symptom then shows the fix:**

In [17]:
import mlx.core as mx

# -- Symptom 1: off-by-one in RoPE position indexing -----------
# Demonstrate that a 1-index shift on BOTH q and k preserves the
# relative-distance property — but a 1-index shift on ONLY ONE
# (the classic off-by-one) silently breaks attention.
import math

d_head = 64
max_pos = 128
j = mx.arange(0, d_head, 2, dtype=mx.float32)
theta = 1.0 / (10000.0 ** (j / d_head))
positions = mx.arange(0, max_pos, dtype=mx.float32)
angles = positions[:, None] * theta[None, :]
cos_tab = mx.cos(angles)
sin_tab = mx.sin(angles)
mx.eval(cos_tab, sin_tab)

def apply_rope(x: mx.array, pos: int) -> mx.array:
    c = cos_tab[pos]; s = sin_tab[pos]
    x_even = x[..., 0::2]; x_odd = x[..., 1::2]
    rot_even = x_even * c - x_odd * s
    rot_odd = x_even * s + x_odd * c
    out = mx.stack([rot_even, rot_odd], axis=-1)
    return out.reshape(x.shape)

mx.random.seed(0)
q = mx.random.normal(shape=(d_head,))
k = mx.random.normal(shape=(d_head,))
# Correct: both q and k rotated at their true positions m, n.
m, n = 10, 25
ref = float(mx.sum(apply_rope(q, m) * apply_rope(k, n)).item())
# Off-by-one on q ONLY: uses position m+1 for q but n for k.
bug = float(mx.sum(apply_rope(q, m + 1) * apply_rope(k, n)).item())
rel_drift = abs(ref - bug) / (abs(ref) + 1e-9)
print(f"[1] correct q·k score at (m={m}, n={n}):      {ref:+.4f}")
print(f"    off-by-one on q (m+1, n): {bug:+.4f}   drift = {rel_drift:.1%}")
assert rel_drift > 0.0, "sanity: off-by-one must perturb the score"
print("    → symptom: long-range attention shifted by 1 token; fix: unify indexing.")

# -- Symptom 2: RoPE freq-base (theta_base) mismatch -----------
# Build (cos, sin) at theta_base=10000 vs 500000 for the SAME position
# and show they're materially different — this is the LLaMA-3.1 bug
# every serving framework guarded against in mid-2024.
def rope_at_pos(pos: int, theta_base: float) -> mx.array:
    thetas = 1.0 / (theta_base ** (j / d_head))
    ang = pos * thetas
    return mx.cos(ang), mx.sin(ang)

pos = 60000  # well within a 128k-context model
c_small, s_small = rope_at_pos(pos, 10000.0)
c_large, s_large = rope_at_pos(pos, 500000.0)
mx.eval(c_small, s_small, c_large, s_large)
# The two (cos, sin) tables diverge substantially at high positions
# on the LOW-frequency end — where 10000-base aliases and 500000-base
# is still in its first cycle.
theta_diff = float(mx.max(mx.abs(c_small - c_large)).item())
print(f"[2] max |cos_10k − cos_500k| at pos={pos}: {theta_diff:.4f}")
assert theta_diff > 0.1, (
    "theta_base mismatch must produce materially different cos tables"
)
print("    → symptom: catastrophic perplexity explosion past training length.")
print("    → fix: load rope_theta from config.json; assert exact match.")

# -- Symptom 3: fp16 / bf16 precision loss in cos, sin ---------
# Compute (cos, sin) in fp32 at a very high position and compare to
# the bf16-native computation. The bf16 path LOSES precision on the
# low-frequency end (where m · θ_j is small but m is large).
pos_hi = 100_000
angle_fp32 = pos_hi * theta
angle_bf16 = (
    mx.array(pos_hi, dtype=mx.bfloat16) * theta.astype(mx.bfloat16)
).astype(mx.float32)
rel_err = float(
    mx.max(mx.abs((angle_fp32 - angle_bf16) / (angle_fp32 + 1e-9))).item()
)
cos_fp32 = mx.cos(angle_fp32)
cos_bf16 = mx.cos(
    mx.array(pos_hi, dtype=mx.bfloat16) * theta.astype(mx.bfloat16)
).astype(mx.float32)
cos_diff = float(mx.max(mx.abs(cos_fp32 - cos_bf16)).item())
mx.eval(cos_fp32, cos_bf16)
print(f"[3] bf16 rope-angle relative error at pos={pos_hi}: {rel_err:.2%}")
print(f"    max |cos_fp32 − cos_bf16|                         = {cos_diff:.4f}")
assert cos_diff >= 0.0, "sanity check"
print("    → fix: precompute (cos, sin) in fp32; cast DOWN to bf16 at use.")


[1] correct q·k score at (m=10, n=25):      -6.4119
    off-by-one on q (m+1, n): -2.4816   drift = 61.3%
    → symptom: long-range attention shifted by 1 token; fix: unify indexing.
[2] max |cos_10k − cos_500k| at pos=60000: 1.7433
    → symptom: catastrophic perplexity explosion past training length.
    → fix: load rope_theta from config.json; assert exact match.
[3] bf16 rope-angle relative error at pos=100000: 0.48%
    max |cos_fp32 − cos_bf16|                         = 1.9837
    → fix: precompute (cos, sin) in fp32; cast DOWN to bf16 at use.


---

### 📋 Interview Question Index

| ID | Difficulty | Roles | Question |
|---|---|---|---|
| `nb04-q01` | warmup | mle | Compare absolute sinusoidal positional encoding (the original Transformer) wi... |
| `nb04-q02` | core | mle, research_engineer | Derive RoPE from first principles. What property do we WANT the position-awar... |
| `nb04-q03` | core | research_engineer, systems_engineer | Explain ALiBi. Write the slope formula for H heads, reason through why it's s... |
| `nb04-q04` | core | research_engineer | What is NoPE (2023) and what's the surprising finding? If transformers are pe... |
| `nb04-q05` | stretch | research_engineer, systems_engineer | A model was trained with RoPE at 4K context. You want to extend to 128K at in... |
| `nb04-q06` | stretch | mle, systems_engineer | Explain the `rope_base` (θ_base, usually 10000) hyperparameter in RoPE. Why d... |
| `nb04-q07` | research | research_engineer, systems_engineer | In Grouped-Query Attention (LLaMA-2/3, Mistral, Qwen) a single K/V head is sh... |